# RAG Evaluation

## pre-requisites

1. Start docker containers for FastAPI, Postgres - `docker compose up`
2. Start Vercel Eve app -- this starts the Eve server - `cd bernalillo-water-rag-agent/ && pnpm run dev`

In [49]:
from utils.utils import (
    DEFAULT_GROUND_TRUTH,
    GroundTruthQuery,
    get_connection,
    load_ground_truth,
)
import pandas as pd
from tqdm.auto import tqdm
from pathlib import Path
import requests
import json

from openai import OpenAI
from typing import cast

df_ground_truth = pd.read_csv('../data/processed/search_ground_truth.csv')

ground_truth = df_ground_truth.to_dict(orient="records")

In [ ]:
def ask_eve__get_session(question: str) -> str | None:
    eve_health_url = "http://127.0.0.1:59119/eve/v1/health"

    health_check = False
    try:
        health_check = requests.get(eve_health_url).status_code == 200
    except:
        print('Eve service doesn\'t appear to be running')
        raise ConnectionError

    if not health_check:
        return None

    eve_session_url = "http://127.0.0.1:59119/eve/v1/session"
    payload = {
        'message': question
    }
    session = None
    try:
        session = requests.post(eve_session_url, json=payload)
    except:
        print('Issue creating Eve session')
        raise ConnectionError
    
    session_json = session.json()
    session_id   = session_json['sessionId']

    return session_id

In [23]:
session_id = ask_eve__get_session("hi")

In [24]:
print(session_id)

wrun_01M08CN4E7BGPKEJZM995A0R5A


In [56]:
from dataclasses import dataclass


@dataclass(frozen=True)
class EveResponse:
    question: str
    document: int
    answer_llm: str
    session_id: str

def reset_eve_session(session_id: str, eve_host: str = "http://127.0.0.1:59119") -> None:
    try:
        requests.post(
            f"{eve_host}/eve/v1/session/{session_id}/reset",
            json={"reason": "eval complete"},
            timeout=10,
        )
    except Exception:
        print(f"Issue resetting Eve session {session_id}")


def ask_eve(ground_truth: GroundTruthQuery) -> EveResponse | None:
    eve_host = "http://127.0.0.1:59119"
    session_id = ask_eve__get_session(ground_truth['question'])

    if not session_id:
        return None
    stream_url = f"{eve_host}/eve/v1/session/{session_id}/stream"
    response = None
    try:
        response = requests.get(stream_url, stream=True)
        response.raise_for_status()

        completed_messages = []
        for line in response.iter_lines(decode_unicode=True):
            if not line:
                continue
            event = json.loads(line)
            if event['type'] == 'message.completed' and event['data']['finishReason'] == 'stop':
                completed_messages.append(event['data']['message'])
            if event['type'] in ('session.waiting', 'session.completed', 'turn.failed'):
                if not completed_messages:
                    return None
                return EveResponse(ground_truth['question'], ground_truth['document'], completed_messages[-1], session_id)
        return None
    except Exception:
        print("error reading eve session stream")
        return None
    finally:
        if response is not None:
            response.close()
        reset_eve_session(session_id, eve_host)


In [52]:
answer_llm = ask_eve(cast(GroundTruthQuery, ground_truth[0]))

In [53]:
print(answer_llm)
print(ground_truth[0])

EveResponse(question='Does Albuquerque drinking water meet federal safety standards?', document=159, answer_llm='Yes—based on the Albuquerque Water Quality reports in the database, the drinking water is reported as meeting the federal standards shown there.\n\nWhy I say that:\n- The reports state EPA limits the amount of certain contaminants in drinking water, and Albuquerque’s reported test results are presented as the latest results for regulated substances.\n- The compliance data for 2020–2025 shows reported levels at or below the listed federal limits where those limits are provided. Examples include:\n  - Gross Alpha Particle Activity: reported max 1.6 pCi/L in 2023–2024, below the MCL of 15 pCi/L.\n  - Total Coliform: reported max 0.41% in 2020–2025, below the MCL of 5%.\n  - Turbidity: reported max 0.11–0.56 NTU in 2021–2025, below the MCL of 1 NTU.\n  - In 2025, E. coli is listed with an MCL of 1 and reported max 0.41%.\n\nSource: https://www.abcwua.org/wp-content/uploads/2026/

In [57]:
def cache_eve_call(ground_truth: GroundTruthQuery, store: list[EveResponse]):
    response = ask_eve(ground_truth)
    if response:
        store.append(response)

In [58]:
store: list[EveResponse] = []
ground_truth_sample = ground_truth[:10]
for truth in tqdm(ground_truth_sample):
    cache_eve_call(cast(GroundTruthQuery,truth), store)

100%|██████████| 10/10 [00:51<00:00,  5.11s/it]


In [61]:
df_eve_responses = pd.DataFrame(store)
print(df_eve_responses)
df_eve_responses.to_csv('agent_evaluation_data.csv', index=False)

                                            question  document  \
0  Does Albuquerque drinking water meet federal s...       159   
1  Is the 2025 report saying our tap water outper...       159   
2  How does ABCWUA treat groundwater before it ge...       161   
3       Do they extra-filter well water for arsenic?       161   
4  Why does river water need more treatment than ...       161   
5  How is water treated at the San Juan-Chama plant?       161   
6  What is chemical stabilization for in the trea...       161   
7  Is Heron Reservoir part of how water gets to A...       162   
8  Does the San Juan-Chama project run through Ne...       162   
9  How many water samples does ABCWUA test each y...       164   

                                          answer_llm  \
0  Yes—based on the Albuquerque Water Authority r...   
1  Yes — the 2025 report says the water is “outpe...   
2  ABCWUA says groundwater generally requires lit...   
3  Yes. The reports say groundwater “requires lit

## Judge

In [ ]:
# read csv
eve_responses = pd.read_csv('agent_evaluation_data_gpt_5_4_mini.csv')

In [69]:
print(eve_responses.head())

                                            question  document  \
0  Does Albuquerque drinking water meet federal s...       159   
1  Is the 2025 report saying our tap water outper...       159   
2  How does ABCWUA treat groundwater before it ge...       161   
3       Do they extra-filter well water for arsenic?       161   
4  Why does river water need more treatment than ...       161   

                                          answer_llm  \
0  Yes—based on the Albuquerque Water Authority r...   
1  Yes — the 2025 report says the water is “outpe...   
2  ABCWUA says groundwater generally requires lit...   
3  Yes. The reports say groundwater “requires lit...   
4  River water needs more treatment because, acco...   

                        session_id  
0  wrun_01M08FFMMGJPS4XBR9A2VT78WN  
1  wrun_01M08FFTBZQFNBSH902WFB8021  
2  wrun_01M08FG1GVGXH3S1ETME2HACFZ  
3  wrun_01M08FG68N8KTNJFXXREAZQSBA  
4  wrun_01M08FGAPP151654D3PV4E5H2C  


In [64]:
prompt2_template = """
You are an expert evaluator for a RAG system.
Your task is to analyze the relevance of the generated answer to the given question.
Based on the relevance of the generated answer, you will classify it
as 'NON_RELEVANT', 'PARTLY_RELEVANT', or 'RELEVANT'.

Here is the data for evaluation:

Question: {question}
Generated Answer: {answer_llm}

Please analyze the content and context of the generated answer in relation to the question
and provide your evaluation in parsable JSON without using code blocks:

{{
  'Relevance': 'NON_RELEVANT' | 'PARTLY_RELEVANT' | 'RELEVANT',
  'Explanation': '[Provide a brief explanation for your evaluation]'
}}
""".strip()

In [ ]:
from dotenv import load_dotenv
from openai import OpenAI
load_dotenv()
openai_client = OpenAI()

sample = eve_responses.to_dict(orient="records")

evaluations = []

for record in tqdm(sample):
    question = record["question"]
    answer_llm = record['answer_llm']

    prompt = prompt2_template.format(
        question=question,
        answer_llm=answer_llm
    )

    evaluation = openai_client.responses.create(model="gpt-5.4-mini", input=prompt)

    evaluations.append((record, json.loads(evaluation.output_text)))

  0%|          | 0/10 [00:00<?, ?it/s]

100%|██████████| 10/10 [00:13<00:00,  1.32s/it]


In [74]:
evaluations[0]

({'question': 'Does Albuquerque drinking water meet federal safety standards?',
  'document': 159,
  'answer_llm': 'Yes—based on the Albuquerque Water Authority reports in this database, the drinking water is reported as meeting federal EPA safety standards in the years shown.\n\nWhy I’m saying that:\n- The reports state that EPA limits contaminants in drinking water and present the latest test results for regulated substances.\n- The measured values shown for reported contaminants are below their listed federal limits/MCLs in the available data, including:\n  - Gross alpha particle activity: 0.4 pCi/L vs MCL 15\n  - Total coliform: 0.41% vs MCL 5\n  - Turbidity: 0.11–0.2 NTU vs MCL 1\n  - In 2025, E. coli is 0.41% vs MCL 1\n\nSource:\n- https://www.abcwua.org/wp-content/uploads/2026/05/ABCWUA-2025WaterQualityMailerWeb.pdf',
  'session_id': 'wrun_01M08FFMMGJPS4XBR9A2VT78WN'},
 '{\n  "Relevance": "PARTLY_RELEVANT",\n  "Explanation": "The answer is directly about Albuquerque drinking wat

In [80]:
df_eval = pd.DataFrame(evaluations, columns=["record", "evaluation"])
categories = ["RELEVANT", "PARTLY_RELEVANT", "NON_RELEVANT"]

df_eval["evaluation"] = df_eval["evaluation"].apply(json.loads)
df_eval["id"] = df_eval.record.apply(lambda d: d["document"])
df_eval["question"] = df_eval.record.apply(lambda d: d["question"])
df_eval["relevance"] = df_eval.evaluation.apply(lambda d: d["Relevance"])
df_eval["explanation"] = df_eval.evaluation.apply(lambda d: d["Explanation"])

df_eval["relevance"] = pd.Categorical(
    df_eval["relevance"],
    categories=categories,
)

df_eval.relevance.value_counts(normalize=True)

relevance
RELEVANT           0.8
PARTLY_RELEVANT    0.2
NON_RELEVANT       0.0
Name: proportion, dtype: float64